# Conectando a LLMs vía API (OpenAI)

En este cuaderno, revisamos cómo conectar e interactuar con LLMs/modelos de embeddings mediante Python. 

Nos enfocamos en la API y modelos de OpenAI (ChatGPT), pero conexión con otros proveedores como Google (Gemini) y Anthropic (Claude) se logra prácticamente de la misma manera. Aún más, recomendamos revisar OpenRouter si se busca mayor libertad a la hora de elegir modelos.

## Manejo de secretos

La manera estándar de conectar con (casi) cualquier API es vía un token/api-key/secret: una cadena de texto larga y complicada que sirve como una contraseña.

Esta api-key está ligada directamente con nuestra cuenta y a los créditos/tarjetas vinculadas con ella, por lo que es muy importante resguardarla muy bien. Si por accidente se filtra, los adversarios podrán utilizar todo el crédito asociado a la cuenta o incluso generar gastos importantísimos en caso de estar en un plan pay-as-you-go.

Dado que los notebooks de colab normalemente se comparten entre equipos o se hacen públicos, recomendamos usar la funcionalidad de secrets de colab (análogo a las variables de entorno) directamente, o usar la librería `getpass` la cual nosotros usaremos.

In [2]:
import getpass

In [4]:
api_key = getpass.getpass("Ingresa tu api key: ")

print("Api key capturada.")

Api key capturada.


## Catálogo de modelos

Revisar todos los modelos disponibles y pricing en: https://developers.openai.com/api/docs/models

## Conexión a la API

### Modelos de lenguaje (LLMs)

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.chat.completions.create(
    model="gpt-5.5",
    messages=[
        {"role": "system", "content": "Eres un asistente útil."},
        {"role": "user", "content": "Hola, ¿cómo estás?"}
    ]
)

print(response.choices[0].message.content)

Para entender mejor cómo funcionan este tipo de APIs, vale la pena revisar: https://openrouter.ai/docs/api/api-reference/chat/send-chat-completion-request

### Modelos de embeddings

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Hola mundo"
)

embedding = response.data[0].embedding

print(len(embedding))
print(embedding[:5])  # primeros 5 valores

**Importante para pricing**
- Los modelos de embeddings generan vectores de dimensión fija.
- Los modelos generativos (chat/completions) generan secuencias de longitud variable.

## ¿Qué está pasando detrás de una conversación?

Idea clave: La API es **stateless**. El modelo no recuerda conversaciones pasadas entre requests.

Por eso, en cada llamada debemos reenviar **todo** el historial relevante.

### Conversaciones = lista de mensajes

```python
messages = [
    {"role": "system", "content": "Eres un tutor de matemáticas."},
    {"role": "user", "content": "¿Qué es una derivada?"},
    {"role": "assistant", "content": "Es una tasa de cambio."},
    {"role": "user", "content": "¿Y geométricamente?"}
]
```

El modelo recibe TODA la lista y genera el siguiente mensaje.


### Roles más comunes

`system`

Define comportamiento global:

```python
{"role": "system", "content": "Responde en español y sé técnico."}
```

---

`user`

Mensaje del usuario:

```python
{"role": "user", "content": "¿Qué es un embedding?"}
```

---

`assistant`

Respuesta previa del modelo:

```python
{"role": "assistant", "content": "Un embedding es un vector semántico."}
```

Esto permite continuidad entre turnos.

---

### La “memoria” realmente funciona así

Primera llamada:

```python
[
    {"role": "user", "content": "Hola"}
]
```

Segunda llamada:

```python
[
    {"role": "user", "content": "Hola"},
    {"role": "assistant", "content": "¡Hola!"},
    {"role": "user", "content": "¿Qué es álgebra lineal?"}
]
```

La API no recordó nada.

Simplemente releímos el historial completo.

---

### Problema: Context rot

Los modelos tienen una ventana de contexto limitada:
- 8k,
- 32k,
- 128k tokens,
- etc.

Mientras más larga la conversación más costo, más latencia y peor enfoque.


En conversaciones largas:
- instrucciones viejas interfieren,
- información importante se “entierra”,
- aparecen contradicciones,
- baja coherencia.

A esto informalmente se le llama **"context rot"**

¡Por eso el manejo de contexto es muy importante!

Las aplicaciones reales suelen:
- eliminar mensajes viejos,
- resumir historial,
- usar RAG/vector DBs,
- mantener prompts importantes en `system`.
